In [ ]:
import pandas as pd
import wandb

# Initialize wandb
wandb.init(project="your_project_name")

# Fetch runs from your project
api = wandb.Api()
runs = api.runs("rt_rank", filters={"display_name": {"$regex": "^RL_.*nfeedback_5000*"}})

In [ ]:
len(runs)

In [92]:
# Create a list to store data from filtered runs
filtered_run_data = []

# Iterate through the runs
for run in runs:
    # Check if the run name starts with "ppo_"
    if run.name.startswith("RL_") and "ensemble" not in run.name:
        # Get the summary statistics (includes final values of metrics)
        summary = run.summary._json_dict

        # Get the history (includes all logged metrics)
        history = run.history(keys=["rollout/ep_rew_mean" if "merge" in run.name else "eval/mean_reward", "global_step"])
        print(history)

        # Combine summary and history data
        run_data = {
            "run_id": run.id,
            "run_name": run.name,
            **summary,
            **{f"{k}_history": v.tolist() for k, v in history.items()}
        }

        filtered_run_data.append(run_data)

"""for run in runs_orig:
    # Check if the run name starts with "ppo_"
    if run.name.startswith("RL_") and "ensemble" not in run.name:
        # Get the summary statistics (includes final values of metrics)
        summary = run.summary._json_dict

        # Get the history (includes all logged metrics)
        history = run.history(keys=["eval/mean_reward", "global_step"])

        # Combine summary and history data
        run_data = {
            "run_id": run.id,
            "run_name": run.name,
            **summary,
            **{f"{k}_history": v.tolist() for k, v in history.items()}
        }

        filtered_run_data.append(run_data)
"""


# Create a DataFrame from filtered run data
orig_df = pd.DataFrame(filtered_run_data)

filtered_df=orig_df

    _step  eval/mean_reward  global_step
0      17         26.997036      10000.0
1      45         -7.283094      20000.0
2      86         25.890026      30000.0
3     114         25.811800      40000.0
4     155         27.239697      50000.0
..    ...               ...          ...
95   3223         33.899540     960000.0
96   3251         34.740410     970000.0
97   3292         38.043106     980000.0
98   3320         34.664950     990000.0
99   3362         37.939415    1000000.0

[100 rows x 3 columns]
    _step  eval/mean_reward  global_step
0     236          1.560423      10000.0
1     498        751.255550      20000.0
2     747        587.710750      30000.0
3    1009        820.153560      40000.0
4    1258        379.023560      50000.0
..    ...               ...          ...
95  24541       5259.216000     960000.0
96  24803       5088.815000     970000.0
97  25065       4728.195000     980000.0
98  25314       4195.260000     990000.0
99  25577       5049.274000    10

In [93]:
filtered_df.to_csv("mujoco_data.csv")

In [95]:
import colorsys
from collections import OrderedDict, defaultdict

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

def get_score_field(env_name: str):
    return "rollout/ep_rew_mean" if "merge" in env_name else "eval/mean_reward"

# Function to extract environment, feedback type, noise level, and RT variant from run name
def extract_info(run_name):
    parts = run_name.split('_')
    
    # Check if it's a noise-free run (starts with "ppo_")
    if run_name.startswith("ppo_"):
        env = parts[1]  # e.g., "Swimmer-v5"
        feedback = parts[3]  # e.g., "comparative"
        noise = 0.0  # No "noise" keyword in name means noise-free
        is_rt = run_name.endswith("rt1.0")
    # Check if it's a noisy run (starts with "RL_ppo_")
    elif run_name.startswith("RL_ppo_"):
        env = parts[2]  # e.g., "Swimmer-v5"
        feedback = parts[4]  # e.g., "comparative"
        # Find noise level (should have "noise" keyword followed by value)
        if "noise" in parts:
            noise_idx = parts.index("noise")
            noise = float(parts[noise_idx + 1])
        else:
            # Fallback, though shouldn't happen for RL_ppo runs
            noise = 0.0
        is_rt = run_name.endswith("rt1.0")
    else:
        return None, None, None, None
    
    return env, feedback, noise, is_rt

def safe_convert_to_float(value):
    try:
        return float(value)
    except (ValueError, TypeError):
        return np.nan

# Group runs by environment, feedback type, noise level, and RT variant
grouped_runs = defaultdict(lambda: defaultdict(lambda: defaultdict(lambda: defaultdict(list))))
for _, row in filtered_df.iterrows():
    env, feedback, noise, is_rt = extract_info(row['run_name'])
    if env is None:  # Skip if parsing failed
        continue
    if isinstance(row[f'{get_score_field(env)}_history'], float):
        continue
    
    # Clean the reward history
    row[f'{get_score_field(env)}_history'] = [np.nan if x == "nan" else x for x in row[f'{get_score_field(env)}_history']]
    
    rt_variant = "rt" if is_rt else "non_rt"
    grouped_runs[env][feedback][noise][rt_variant].append(row)

# Define colors for RT variants
colors = {
    'rt': '#2ca02c',      # green
    'non_rt': '#1f77b4'   # blue
}

# Plotting function for each environment
def plot_environment_comparison(env, feedback_runs):
    # Get all available feedback types for this environment
    feedback_types = list(feedback_runs.keys())
    noise_levels = [0.0, 0.1, 0.25, 0.5]

    n_feedback = len(feedback_types)

    # For a 2x2 grid per feedback type:
    # total rows = 2 rows per feedback type
    n_rows = n_feedback * 2
    n_cols = 2

    # Wider than tall; height scales with number of feedback types
    fig, axes = plt.subplots(n_rows, n_cols, figsize=(12, 6 * n_feedback))
    fig.suptitle(f"RL Training Curves for {env}: RT vs Non-RT Comparison", fontsize=16)

    # When n_feedback == 1, axes is shape (2,2); otherwise it's (n_feedback*2, 2)
    # We'll index it consistently as axes[row, col]
    for fb_idx, feedback in enumerate(feedback_types):
        for noise_idx, noise in enumerate(noise_levels):
            # Map noise index 0..3 into a 2x2 grid:
            #   0 -> (0,0), 1 -> (0,1), 2 -> (1,0), 3 -> (1,1)
            row_base = fb_idx * 2
            grid_r = row_base + (noise_idx // 2)
            grid_c = noise_idx % 2
            ax = axes[grid_r, grid_c] if n_feedback > 1 else axes[grid_r, grid_c]

            # Plot both RT and non-RT variants for this feedback type and noise level
            for rt_variant in ['rt', 'non_rt']:
                if rt_variant not in feedback_runs[feedback][noise]:
                    continue

                runs = feedback_runs[feedback][noise][rt_variant]
                if not runs:
                    continue

                # Find the maximum number of steps across all runs
                max_steps = 0
                for run in runs:
                    steps = [safe_convert_to_float(step) for step in run['global_step_history']]
                    valid_steps = [s for s in steps if not np.isnan(s)]
                    if valid_steps:
                        max_steps = max(max_steps, max(valid_steps))

                if max_steps == 0:
                    continue

                # Create a common x-axis (steps)
                common_steps = np.arange(0, int(max_steps) + 1, 1000)
                all_rewards = np.full((len(runs), len(common_steps)), np.nan)

                for i, run in enumerate(runs):
                    steps = np.array([safe_convert_to_float(step) for step in run['global_step_history']])
                    rewards = np.array([safe_convert_to_float(reward) for reward in run[f'{get_score_field(env)}_history']])

                    # Remove any NaN values
                    valid = ~np.isnan(steps) & ~np.isnan(rewards)
                    steps = steps[valid]
                    rewards = rewards[valid]

                    if len(steps) > 0 and len(rewards) > 0:
                        # Interpolate the rewards to the common step range
                        interpolated_rewards = np.interp(common_steps, steps, rewards)
                        all_rewards[i] = interpolated_rewards

                # Calculate mean and std
                mean_reward = np.nanmean(all_rewards, axis=0)
                std_reward = np.nanstd(all_rewards, axis=0)

                # Plot the mean with confidence interval
                label = f"{'RT' if rt_variant == 'rt' else 'Non-RT'}"
                color = colors[rt_variant]

                ax.plot(common_steps, mean_reward, label=label, color=color, linewidth=2)
                ax.fill_between(common_steps, mean_reward - std_reward, mean_reward + std_reward,
                                color=color, alpha=0.2)

            # Set subplot properties
            ax.set_title(f"{feedback.capitalize()}, Noise: {noise}", fontsize=12)
            ax.set_xlabel("Global Steps", fontsize=10)
            ax.set_ylabel("Mean Reward", fontsize=10)
            ax.tick_params(axis='both', which='major', labelsize=9)
            ax.legend(fontsize=9)
            ax.grid(True, alpha=0.3)

    # Synchronize y-axes across all subplots for this environment
    all_ylims = []
    for r in range(n_rows):
        for c in range(n_cols):
            ylim = axes[r, c].get_ylim() if n_feedback > 1 else axes[r, c].get_ylim()
            all_ylims.extend(ylim)

    if all_ylims:
        global_ymin, global_ymax = min(all_ylims), max(all_ylims)
        for r in range(n_rows):
            for c in range(n_cols):
                (axes[r, c] if n_feedback > 1 else axes[r, c]).set_ylim(global_ymin, global_ymax)

    plt.tight_layout()
    plt.subplots_adjust(top=0.90, hspace=0.35, wspace=0.25)
    out_path = f"rt_comparison_{env}.png"
    plt.savefig(out_path, dpi=300, bbox_inches='tight')
    plt.close()
    print(f"RT comparison plots for {env} have been saved to {out_path}")


# Create comparison plots for each environment
for env, feedback_runs in grouped_runs.items():
    plot_environment_comparison(env, feedback_runs)

# Also create a summary function to show what data we have
def print_data_summary():
    print("Data Summary:")
    print("=" * 50)
    for env in grouped_runs:
        print(f"\nEnvironment: {env}")
        for feedback in grouped_runs[env]:
            print(f"  Feedback: {feedback}")
            for noise in sorted(grouped_runs[env][feedback].keys()):
                rt_counts = len(grouped_runs[env][feedback][noise].get('rt', []))
                non_rt_counts = len(grouped_runs[env][feedback][noise].get('non_rt', []))
                print(f"    Noise {noise}: RT={rt_counts} runs, Non-RT={non_rt_counts} runs")

# Print summary of available data
print_data_summary()

RT comparison plots for Swimmer-v5 have been saved to rt_comparison_Swimmer-v5.png
RT comparison plots for HalfCheetah-v5 have been saved to rt_comparison_HalfCheetah-v5.png
RT comparison plots for Walker2d-v5 have been saved to rt_comparison_Walker2d-v5.png
Data Summary:

Environment: Swimmer-v5
  Feedback: comparative
    Noise 0.0: RT=5 runs, Non-RT=5 runs
    Noise 0.1: RT=5 runs, Non-RT=5 runs
    Noise 0.25: RT=5 runs, Non-RT=5 runs
    Noise 0.5: RT=5 runs, Non-RT=5 runs

Environment: HalfCheetah-v5
  Feedback: comparative
    Noise 0.0: RT=5 runs, Non-RT=5 runs
    Noise 0.1: RT=5 runs, Non-RT=5 runs
    Noise 0.25: RT=5 runs, Non-RT=5 runs
    Noise 0.5: RT=5 runs, Non-RT=5 runs

Environment: Walker2d-v5
  Feedback: comparative
    Noise 0.0: RT=5 runs, Non-RT=5 runs
    Noise 0.1: RT=5 runs, Non-RT=5 runs
    Noise 0.25: RT=5 runs, Non-RT=5 runs
    Noise 0.5: RT=5 runs, Non-RT=5 runs


In [30]:
import pandas as pd
import numpy as np
from collections import defaultdict

def safe_convert_to_float(value):
    try:
        return float(value)
    except (ValueError, TypeError):
        return np.nan

def extract_info(run_name):
    """Extract environment, feedback type, noise level, and RT variant from run name"""
    parts = run_name.split('_')
    
    # Check if it's a noise-free run (starts with "ppo_")
    if run_name.startswith("ppo_"):
        env = parts[1]  # e.g., "Swimmer-v5"
        feedback = parts[3]  # e.g., "comparative"
        noise = 0.0  # No "noise" keyword in name means noise-free
        is_rt = run_name.endswith("rt1.0")
    # Check if it's a noisy run (starts with "RL_ppo_")
    elif run_name.startswith("RL_ppo_"):
        env = parts[2]  # e.g., "Swimmer-v5"
        feedback = parts[4]  # e.g., "comparative"
        # Find noise level (should have "noise" keyword followed by value)
        if "noise" in parts:
            noise_idx = parts.index("noise")
            noise = float(parts[noise_idx + 1])
        else:
            # Fallback, though shouldn't happen for RL_ppo runs
            noise = 0.0
        is_rt = run_name.endswith("rt1.0")
    else:
        return None, None, None, None
    
    return env, feedback, noise, is_rt

def get_max_and_final_reward(run_data, score_field="rollout/ep_rew_mean"):
    """Extract max and final reward from a run's history"""
    if isinstance(run_data[f'{score_field}_history'], float):
        return np.nan, np.nan
    
    # Clean the reward history
    rewards = [safe_convert_to_float(x) for x in run_data[f'{score_field}_history']]
    rewards = [r for r in rewards if not np.isnan(r)]
    
    if not rewards:
        return np.nan, np.nan
    
    max_reward = max(rewards)
    final_reward = rewards[-1]  # Last reward in the history
    
    return max_reward, final_reward

def create_results_table(filtered_df, score_field="rollout/ep_rew_mean"):
    """Create a markdown table comparing RT vs non-RT results"""
    
    # Group runs by environment, feedback type, noise level, and RT variant
    grouped_runs = defaultdict(lambda: defaultdict(lambda: defaultdict(lambda: defaultdict(list))))
    
    for _, row in filtered_df.iterrows():
        env, feedback, noise, is_rt = extract_info(row['run_name'])
        if env is None:  # Skip if parsing failed
            continue
        if isinstance(row[f'{score_field}_history'], float):
            continue
        
        rt_variant = "rt" if is_rt else "non_rt"
        grouped_runs[env][feedback][noise][rt_variant].append(row)
    
    # Calculate statistics for each group
    results = {}
    
    for env in grouped_runs:
        for feedback in grouped_runs[env]:
            for noise in grouped_runs[env][feedback]:
                key = f"{env}_{feedback}_noise{noise}"
                results[key] = {}
                
                for rt_variant in ['rt', 'non_rt']:
                    if rt_variant not in grouped_runs[env][feedback][noise]:
                        results[key][rt_variant] = {'max_mean': np.nan, 'max_std': np.nan, 
                                                  'final_mean': np.nan, 'final_std': np.nan}
                        continue
                    
                    runs = grouped_runs[env][feedback][noise][rt_variant]
                    max_rewards = []
                    final_rewards = []
                    
                    for run in runs:
                        max_rew, final_rew = get_max_and_final_reward(run, score_field)
                        if not np.isnan(max_rew):
                            max_rewards.append(max_rew)
                        if not np.isnan(final_rew):
                            final_rewards.append(final_rew)
                    
                    # Calculate mean and std
                    results[key][rt_variant] = {
                        'max_mean': np.mean(max_rewards) if max_rewards else np.nan,
                        'max_std': np.std(max_rewards) if max_rewards else np.nan,
                        'final_mean': np.mean(final_rewards) if final_rewards else np.nan,
                        'final_std': np.std(final_rewards) if final_rewards else np.nan,
                        'n_runs': len(runs)
                    }
    
    # Create markdown table
    markdown_lines = []
    markdown_lines.append("| Environment | Max Reward | Final Reward |")
    markdown_lines.append("|-------------|------------|--------------|")
    
    for key in sorted(results.keys()):
        if 'rt' not in results[key] or 'non_rt' not in results[key]:
            continue
            
        rt_data = results[key]['rt']
        non_rt_data = results[key]['non_rt']
        
        # Skip if we don't have data for both variants
        if (np.isnan(rt_data['max_mean']) or np.isnan(non_rt_data['max_mean']) or 
            np.isnan(rt_data['final_mean']) or np.isnan(non_rt_data['final_mean'])):
            continue
        
        # Calculate differences (RT - non_RT, since non_RT is baseline)
        max_diff = rt_data['max_mean'] - non_rt_data['max_mean']
        final_diff = rt_data['final_mean'] - non_rt_data['final_mean']
        
        # Format the differences with bold for positive values
        max_diff_str = f"**({max_diff:+.1f})**" if max_diff > 0 else f"({max_diff:+.1f})"
        final_diff_str = f"**({final_diff:+.1f})**" if final_diff > 0 else f"({final_diff:+.1f})"
        
        # Format the main values
        max_reward_str = f"{rt_data['max_mean']:.1f} {max_diff_str}"
        final_reward_str = f"{rt_data['final_mean']:.1f} {final_diff_str}"
        
        # Clean up the key for display
        display_key = key.replace('_', ' ').replace('noise', 'noise=')
        
        markdown_lines.append(f"| {display_key} | {max_reward_str} | {final_reward_str} |")
    
    return "\n".join(markdown_lines)

def create_detailed_results_table(filtered_df, score_field="rollout/ep_rew_mean"):
    """Create a more detailed markdown table with separate columns for RT and non-RT"""
    
    # Group runs by environment, feedback type, noise level, and RT variant
    grouped_runs = defaultdict(lambda: defaultdict(lambda: defaultdict(lambda: defaultdict(list))))
    
    for _, row in filtered_df.iterrows():
        env, feedback, noise, is_rt = extract_info(row['run_name'])
        if env is None:  # Skip if parsing failed
            continue
        if isinstance(row[f'{score_field}_history'], float):
            continue
        
        rt_variant = "rt" if is_rt else "non_rt"
        grouped_runs[env][feedback][noise][rt_variant].append(row)
    
    # Calculate statistics for each group
    results = {}
    
    for env in grouped_runs:
        for feedback in grouped_runs[env][feedback]:
            for noise in grouped_runs[env][feedback]:
                key = f"{env}_{feedback}_noise{noise}"
                results[key] = {}
                
                for rt_variant in ['rt', 'non_rt']:
                    if rt_variant not in grouped_runs[env][feedback][noise]:
                        results[key][rt_variant] = {'max_mean': np.nan, 'max_std': np.nan, 
                                                  'final_mean': np.nan, 'final_std': np.nan, 'n_runs': 0}
                        continue
                    
                    runs = grouped_runs[env][feedback][noise][rt_variant]
                    max_rewards = []
                    final_rewards = []
                    
                    for run in runs:
                        max_rew, final_rew = get_max_and_final_reward(run, score_field)
                        if not np.isnan(max_rew):
                            max_rewards.append(max_rew)
                        if not np.isnan(final_rew):
                            final_rewards.append(final_rew)
                    
                    # Calculate mean and std
                    results[key][rt_variant] = {
                        'max_mean': np.mean(max_rewards) if max_rewards else np.nan,
                        'max_std': np.std(max_rewards) if max_rewards else np.nan,
                        'final_mean': np.mean(final_rewards) if final_rewards else np.nan,
                        'final_std': np.std(final_rewards) if final_rewards else np.nan,
                        'n_runs': len(runs)
                    }
    
    # Create detailed markdown table
    markdown_lines = []
    markdown_lines.append("| Environment | RT Max | RT Final | Non-RT Max | Non-RT Final | Max Diff | Final Diff |")
    markdown_lines.append("|-------------|---------|----------|------------|--------------|----------|------------|")
    
    for key in sorted(results.keys()):
        if 'rt' not in results[key] or 'non_rt' not in results[key]:
            continue
            
        rt_data = results[key]['rt']
        non_rt_data = results[key]['non_rt']
        
        # Skip if we don't have data for both variants
        if (np.isnan(rt_data['max_mean']) or np.isnan(non_rt_data['max_mean']) or 
            np.isnan(rt_data['final_mean']) or np.isnan(non_rt_data['final_mean'])):
            continue
        
        # Calculate differences (RT - non_RT, since non_RT is baseline)
        max_diff = rt_data['max_mean'] - non_rt_data['max_mean']
        final_diff = rt_data['final_mean'] - non_rt_data['final_mean']
        
        # Format the differences with bold for positive values
        max_diff_str = f"**{max_diff:+.1f}**" if max_diff > 0 else f"{max_diff:+.1f}"
        final_diff_str = f"**{final_diff:+.1f}**" if final_diff > 0 else f"{final_diff:+.1f}"
        
        # Format the main values with standard deviations
        rt_max_str = f"{rt_data['max_mean']:.1f}±{rt_data['max_std']:.1f}"
        rt_final_str = f"{rt_data['final_mean']:.1f}±{rt_data['final_std']:.1f}"
        non_rt_max_str = f"{non_rt_data['max_mean']:.1f}±{non_rt_data['max_std']:.1f}"
        non_rt_final_str = f"{non_rt_data['final_mean']:.1f}±{non_rt_data['final_std']:.1f}"
        
        # Clean up the key for display
        display_key = key.replace('_', ' ').replace('noise', 'noise=')
        
        markdown_lines.append(f"| {display_key} | {rt_max_str} | {rt_final_str} | {non_rt_max_str} | {non_rt_final_str} | {max_diff_str} | {final_diff_str} |")
    
    return "\n".join(markdown_lines)

# Usage example:
# Assuming you have your filtered_df ready from the wandb data fetching code

# Create the comparison table (RT vs baseline)
print("## RT vs Non-RT Comparison Table")
print(create_results_table(filtered_df, score_field=get_score_field(env_name)))

print("\n" + "="*80 + "\n")

# Create the detailed table
print("## Detailed Results Table")
print(create_detailed_results_table(filtered_df))

## RT vs Non-RT Comparison Table


NameError: name 'env_name' is not defined

In [12]:
orig_df

,run_id,run_name,_runtime,_step,_timestamp,_wandb,epoch,train_loss_epoch,train_loss_step,trainer/global_step,val_accuracy,val_loss,train_pref_loss_epoch,train_pref_loss_step,train_rtrank_loss_epoch,train_rtrank_loss_step
0,qq7xz2r8,ppo_merge-v0_1789_comparative_1789_nfeedback_5000,164,15989,1.766164e+09,{'runtime': 164},29,0.345172,0.935233,15929,0.860215,0.345835,NaN,NaN,NaN,NaN
1,o1urifdi,ppo_merge-v0_1789_comparative_1789_nfeedback_5...,366,18809,1.766164e+09,{'runtime': 366},29,1.182743,1.268545,18749,0.861333,0.586627,0.382448,0.443533,1.182743,1.268545
2,u3dz2bwh,ppo_merge-v0_330_comparative_330_nfeedback_5000,183,15989,1.766164e+09,{'runtime': 183},29,0.351101,0.311882,15929,0.841398,0.356887,NaN,NaN,NaN,NaN
3,cwuesvoz,ppo_merge-v0_330_comparative_330_nfeedback_500...,402,18809,1.766164e+09,{'runtime': 402},29,1.222012,1.391796,18749,0.834667,0.619082,0.396706,0.419302,1.222012,1.391796
4,lntavvji,ppo_merge-v0_912391_comparative_912391_nfeedba...,402,18809,1.766164e+09,{'runtime': 402},29,1.214746,1.322961,18749,0.848000,0.595755,0.396548,0.622454,1.214746,1.322961
5,n9n77dk1,ppo_merge-v0_912391_comparative_912391_nfeedba...,184,15989,1.766164e+09,{'runtime': 184},29,0.361248,0.327476,15929,0.853495,0.357015,NaN,NaN,NaN,NaN
6,ksj2tz8h,ppo_merge-v0_1789_comparative_1789_noise_0.1_n...,165,15989,1.766164e+09,{'runtime': 165},29,0.367628,0.910460,15929,0.840054,0.372502,NaN,NaN,NaN,NaN
7,xt0j9h6v,ppo_merge-v0_1789_comparative_1789_noise_0.1_n...,364,18809,1.766164e+09,{'runtime': 364},29,1.182316,1.249492,18749,0.845333,0.598874,0.395834,0.442461,1.182316,1.249492
8,12m125ut,ppo_merge-v0_912391_comparative_912391_noise_0...,400,18809,1.766164e+09,{'runtime': 400},29,1.215596,1.262801,18749,0.841333,0.600215,0.408535,0.669056,1.215596,1.262801
9,2fgbo4dn,ppo_merge-v0_12_comparative_12_nfeedback_5000_...,377,18809,1.766164e+09,{'runtime': 377},29,1.225095,0.996164,18749,0.825333,0.632914,0.416619,0.437687,1.225095,0.996164


| Environment | Max Reward | Final Reward |
|------------:|------------:|------------:|
| HalfCheetah-v5 noise=0.0 | 5866.9 **(+352.3)** | 5215.2 (-219.3) |
| HalfCheetah-v5 noise=0.1 | 5208.3 (-124.3) | 4922.7 (-387.7) |
| HalfCheetah-v5 noise=0.25 | 5850.7 **(+418.2)** | 5481.4 **(+201.4)** |
| HalfCheetah-v5 noise=0.5 | 5433.3 **(+155.5)** | 5433.3 **(+439.5)** |
| Swimmer-v5 noise=0.0 | 110.0 **(+66.1)** | 98.3 **(+77.1)** |
| Swimmer-v5 noise=0.1 | 94.6 **(+48.9)** | 81.8 **(+49.6)** |
| Swimmer-v5 noise=0.25 | 36.7 (-4.8) | 9.5 (-12.6) |
| Swimmer-v5 noise=0.5 | 47.4 (-29.1) | 17.0 (-49.7) |
| Walker2d-v5 noise=0.0 | 3905.1 **(+1088.1)** | 2679.7 **(+313.2)** |
| Walker2d-v5 noise=0.1 | 3126.1 **(+298.1)** | 2742.2 **(+216.0)** |
| Walker2d-v5 noise=0.25 | 2567.7 (-1344.8) | 1919.1 (-1616.6) |
| Walker2d-v5 noise=0.5 | 1754.5 (-1387.6) | 1621.9 (-1037.9) |
| merge-v0 noise=0.0 | 12.2 **(+1.2)** | 11.4 **(+1.0)** |
| merge-v0 noise=0.1 | 12.1 **(+0.9)** | 11.4 **(+0.9)** |
| merge-v0 noise=0.25 | 11.8 **(+0.5)** | 11.7 **(+1.1)** |
| merge-v0 noise=0.5 | 12.4 **(+0.8)** | 11.5 **(+0.3)** |

| Environment | Max Reward | Final Reward |
|-------------:|------------:|--------------:|
| HalfCheetah-v5 | 5866.9 **(+352.3)** | 5215.2 (-219.3) |
| Swimmer-v5 | 110.0 **(+66.1)** | 98.3 **(+77.1)** |
| Walker2d-v5 | 3905.1 **(+1088.1)** | 2679.7 **(+313.2)** |
| merge-v0 | 12.2 **(+1.2)** | 11.4 **(+1.0)** |

| Environment | Max Reward | Final Reward |
|-------------|------------|--------------|
| merge-v0 noise=0.0 | 12.2 **(+1.2)** | 11.4 **(+1.0)** |
| merge-v0 noise=0.1 | 12.1 **(+0.9)** | 11.4 **(+0.9)** |
| merge-v0 noise=0.25 | 11.8 **(+0.5)** | 11.7 **(+1.1)** |
| merge-v0 noise=0.5 | 12.4 **(+0.8)** | 11.5 **(+0.3)** |

| Environment | Max Reward | Final Reward |
|-------------|------------|--------------|
| merge-v0 comparative noise=0.0 | 12.2 **(+1.2)** | 11.4 **(+1.0)** |
| merge-v0 comparative noise=0.1 | 12.1 **(+0.9)** | 11.4 **(+0.9)** |
| merge-v0 comparative noise=0.25 | 11.8 **(+0.5)** | 11.7 **(+1.1)** |
| merge-v0 comparative noise=0.5 | 12.4 **(+0.8)** | 11.5 **(+0.3)** |

## Reward Model Accuracy

In [ ]:
import pandas as pd
import wandb

# Fetch runs from your project
api = wandb.Api()

# Initialize wandb
wandb.init(project="your_project_name")

# Create a list to store data from filtered runs
filtered_run_data = []

runs = api.runs("rt_rank", filters={"display_name": {"$regex": "^ppo_.*"}})

# Iterate through the runs
for run in runs:
    # Check if the run name starts with "ppo_"
    if run.name.startswith("ppo_") and "ensemble" not in run.name:
        # Get the summary statistics (includes final values of metrics)
        summary = run.summary._json_dict

        # Get the history (includes all logged metrics)
        #history = run.history(keys=["val_accuracy"])

        # Combine summary and history data
        run_data = {
            "run_id": run.id,
            "run_name": run.name,
            **summary,
            #**{f"{k}_history": v.tolist() for k, v in history.items()}
        }

        filtered_run_data.append(run_data)

"""for run in runs_orig:
    # Check if the run name starts with "ppo_"
    if run.name.startswith("RL_") and "ensemble" not in run.name:
        # Get the summary statistics (includes final values of metrics)
        summary = run.summary._json_dict

        # Get the history (includes all logged metrics)
        history = run.history(keys=["eval/mean_reward", "global_step"])

        # Combine summary and history data
        run_data = {
            "run_id": run.id,
            "run_name": run.name,
            **summary,
            **{f"{k}_history": v.tolist() for k, v in history.items()}
        }

        filtered_run_data.append(run_data)
"""


# Create a DataFrame from filtered run data
orig_df = pd.DataFrame(filtered_run_data)

In [71]:
acc_df = orig_df[orig_df['run_name'].str.contains('nfeedback_5000', na=False)]

In [72]:
acc_df.to_csv("mujoco_rew_model_data.csv")

In [79]:
import pandas as pd
filtered_rew_df = pd.read_csv("mujoco_rew_model_data.csv")

In [81]:
filtered_rew_df

,Unnamed: 0,run_id,run_name,_runtime,_step,_timestamp,_wandb,epoch,train_loss_epoch,train_loss_step,train_pref_loss_epoch,train_pref_loss_step,train_rtrank_loss_epoch,train_rtrank_loss_step,trainer/global_step,val_accuracy,val_loss
0,0,0tn1c1f9,ppo_Walker2d-v5_912391_comparative_912391_nois...,138,8009,1.753892e+09,{'runtime': 138},29,1.085808,1.182012,0.331954,0.320717,1.085808,1.182012,7949,0.876359,1.079105
1,3,jgeyfdtl,ppo_Walker2d-v5_912391_comparative_912391_nois...,84,6674,1.753892e+09,{'runtime': 84},24,0.276248,0.268194,NaN,NaN,NaN,NaN,6624,0.881793,0.279065
2,11,6ynj3jnu,ppo_Swimmer-v5_912391_comparative_912391_noise...,97,8009,1.753892e+09,{'runtime': 97},29,0.374088,0.273855,NaN,NaN,NaN,NaN,7949,0.831522,0.372466
3,12,f620ea4e,ppo_Swimmer-v5_912391_comparative_912391_noise...,102,8009,1.753892e+09,{'runtime': 102},29,0.495267,0.421607,NaN,NaN,NaN,NaN,7949,0.752717,0.490125
4,16,9apg1hky,ppo_Swimmer-v5_912391_comparative_912391_noise...,140,8009,1.753892e+09,{'runtime': 140},29,1.313482,1.299182,0.360508,0.305023,1.313482,1.299182,7949,0.842391,1.318996
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
115,339,2aefg50x,ppo_Walker2d-v5_912391_comparative_912391_nois...,103,4388,1.766252e+09,{'runtime': 103},6,0.398209,0.479679,0.182746,0.000645,0.398209,0.479679,4374,0.945333,0.225533
116,340,gvgxjf2x,ppo_Walker2d-v5_1789_comparative_1789_noise_0....,82,7994,1.766252e+09,{'runtime': 82},14,0.109617,0.000947,NaN,NaN,NaN,NaN,7964,0.965054,0.100460
117,341,ogl72pwx,ppo_Walker2d-v5_1789_comparative_1789_noise_0....,105,5015,1.766252e+09,{'runtime': 105},7,0.351439,0.202634,0.172021,0.002204,0.351439,0.202634,4999,0.969333,0.150705
118,342,uya7r7mu,ppo_Walker2d-v5_912391_comparative_912391_nois...,64,5862,1.766252e+09,{'runtime': 64},10,0.124962,0.029392,NaN,NaN,NaN,NaN,5840,0.939516,0.156605


In [83]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from collections import defaultdict

# --- If you already defined extract_info above, you can reuse it and skip this redefinition. ---
def extract_info(run_name: str):
    print(run_name)
    parts = run_name.split('_')
    if run_name.startswith("ppo_"):
        env = parts[1]
        feedback = parts[3]
        if "noise" in parts:
            noise_idx = parts.index("noise")
            noise = float(parts[noise_idx + 1])
        else:
            noise = 0.0
        is_rt = run_name.endswith("rt1.0")
    else:
        return None, None, None, None
    print(env, feedback, noise, is_rt)
    return env, feedback, noise, is_rt

def coerce_numeric(x):
    try:
        return float(x)
    except Exception:
        return np.nan

# 1) Build a tidy accuracy DataFrame -----------------------------------------
def build_accuracy_df(filtered_df: pd.DataFrame) -> pd.DataFrame:
    rows = []
    for _, r in filtered_df.iterrows():
        env, feedback, noise, is_rt = extract_info(str(r["run_name"]))
        if env is None:
            continue
        acc = coerce_numeric(r.get("val_accuracy", np.nan))
        if np.isnan(acc):
            continue
        rows.append({
            "run_id": r.get("run_id"),
            "run_name": r.get("run_name"),
            "env": env,
            "feedback": feedback,
            "noise": float(noise),
            "rt_variant": "rt" if is_rt else "non_rt",
            "val_accuracy": acc,
        })
    acc_df = pd.DataFrame(rows)
    return acc_df

# Optional: restrict ordering
noise_levels = [0.0, 0.1, 0.25, 0.5]
#if "noise" in acc_df.columns:
#    acc_df["noise"] = pd.Categorical(acc_df["noise"], categories=noise_levels, ordered=True)

# 2) Summary tables -----------------------------------------------------------
def make_summary(acc_df: pd.DataFrame):
    # Group and aggregate
    grouped = (acc_df
               .groupby(["env", "feedback", "noise", "rt_variant"])
               .agg(mean_acc=("val_accuracy", "mean"),
                    std_acc=("val_accuracy", "std"),
                    n_runs=("val_accuracy", "count"))
               .reset_index())

    # Pivot for RT vs Non-RT comparisons
    pivot = grouped.pivot(index=["env", "feedback", "noise"],
                          columns="rt_variant",
                          values=["mean_acc", "std_acc", "n_runs"])
    # flatten MultiIndex columns
    pivot.columns = [f"{a}_{b}" for a, b in pivot.columns]
    pivot = pivot.reset_index()

    # Compute diffs (RT - Non-RT)
    pivot["diff_mean"] = pivot.get("mean_acc_rt", np.nan) - pivot.get("mean_acc_non_rt", np.nan)
    return grouped, pivot

def print_compact_accuracy_table(pivot_acc: pd.DataFrame):
    lines = []
    lines.append("| Environment / Feedback / noise | RT mean | Non-RT mean | Δ RT–Non-RT |")
    lines.append("|---|---:|---:|---:|")
    # Sort for nice reading
    ordered = pivot_acc.sort_values(["env", "feedback", "noise"])
    for _, r in ordered.iterrows():
        if pd.isna(r.get("mean_acc_rt")) or pd.isna(r.get("mean_acc_non_rt")):
            continue
        diff = r["diff_mean"]
        diff_str = f"**{diff:+.3f}**" if diff > 0 else f"{diff:+.3f}"
        label = f"{r['env']} / {r['feedback']} / noise={r['noise']}"
        lines.append(f"| {label} | {r['mean_acc_rt']:.3f} | {r['mean_acc_non_rt']:.3f} | {diff_str} |")
    print("\n".join(lines))

def print_detailed_accuracy_table(pivot_acc: pd.DataFrame):
    lines = []
    lines.append("| Environment / Feedback / noise | RT (mean±std, n) | Non-RT (mean±std, n) | Δ RT–Non-RT |")
    lines.append("|---|---:|---:|---:|")
    ordered = pivot_acc.sort_values(["env", "feedback", "noise"])
    for _, r in ordered.iterrows():
        if pd.isna(r.get("mean_acc_rt")) or pd.isna(r.get("mean_acc_non_rt")):
            continue
        rt_str = f"{r['mean_acc_rt']:.3f}±{(r.get('std_acc_rt') or 0):.3f}, n={int(r.get('n_runs_rt') or 0)}"
        nrt_str = f"{r['mean_acc_non_rt']:.3f}±{(r.get('std_acc_non_rt') or 0):.3f}, n={int(r.get('n_runs_non_rt') or 0)}"
        diff = r["diff_mean"]
        diff_str = f"**{diff:+.3f}**" if diff > 0 else f"{diff:+.3f}"
        label = f"{r['env']} / {r['feedback']} / noise={r['noise']}"
        lines.append(f"| {label} | {rt_str} | {nrt_str} | {diff_str} |")
    print("\n".join(lines))


def plot_accuracy_bars(acc_df: pd.DataFrame, save_prefix="rm_accuracy"):
    if acc_df.empty:
        print("No accuracy data to plot.")
        return
    for env, env_df in acc_df.groupby("env"):
        # Order feedback types alphabetically (or keep as encountered)
        feedbacks = sorted(env_df["feedback"].dropna().unique().tolist())
        noises = [n for n in noise_levels if n in env_df["noise"].unique().tolist()]

        # Build a small table for plotting
        g = (env_df.groupby(["feedback", "noise", "rt_variant"])
                    .agg(mean_acc=("val_accuracy", "mean"),
                         std_acc=("val_accuracy", "std"),
                         n_runs=("val_accuracy", "count"))
                    .reset_index())

        # Figure layout: rows = feedback types, cols = noise levels
        n_rows = len(feedbacks)
        n_cols = max(1, len(noises))
        fig, axes = plt.subplots(n_rows, n_cols, figsize=(4*n_cols, 2.8*n_rows), squeeze=False)
        fig.suptitle(f"Reward Model Validation Accuracy — {env}", fontsize=14)

        for i, fb in enumerate(feedbacks):
            for j, nz in enumerate(noises):
                ax = axes[i, j]
                sub = g[(g["feedback"] == fb) & (g["noise"] == nz)]
                # Ensure both variants exist in consistent order
                order = ["non_rt", "rt"]
                means = []
                stds = []
                ns = []
                labels = []
                bar_colors = []
                for variant in order:
                    row = sub[sub["rt_variant"] == variant]
                    if row.empty:
                        means.append(np.nan)
                        stds.append(0.0)
                        ns.append(0)
                    else:
                        means.append(float(row["mean_acc"].values[0]))
                        stds.append(float(row["std_acc"].fillna(0).values[0]))
                        ns.append(int(row["n_runs"].values[0]))
                    labels.append("Non-RT" if variant == "non_rt" else "RT")
                    bar_colors.append(colors[variant])

                x = np.arange(len(order))
                ax.bar(x, means, yerr=stds, capsize=4, width=0.6, color=bar_colors)
                ax.set_xticks(x, labels, fontsize=9)
                ax.set_ylim(0.0, 1.0)  # accuracies in [0,1]; adjust if yours are percentages
                ax.set_ylabel("Val. Accuracy", fontsize=9)
                ax.set_title(f"{fb}, noise={nz} (n={ns[0]}/{ns[1]})", fontsize=10)
                ax.grid(axis="y", alpha=0.3)

        plt.tight_layout()
        plt.subplots_adjust(top=0.90, hspace=0.4, wspace=0.25)
        out_path = f"{save_prefix}_{env}.png"
        plt.savefig(out_path, dpi=300, bbox_inches="tight")
        plt.close()
        print(f"Saved: {out_path}")

# Figures: per-environment bar plots with error bars -----------------------
colors = {
    "rt": "#2ca02c",      # green
    "non_rt": "#1f77b4",  # blue
}

filtered_rew_df = build_accuracy_df(filtered_rew_df)
grouped_acc, pivot_acc = make_summary(filtered_rew_df)

print("## Reward model accuracy — compact comparison (RT vs Non-RT)\n")
print_compact_accuracy_table(pivot_acc)
print("\n" + "="*80 + "\n")
print("## Reward model accuracy — detailed table\n")
print_detailed_accuracy_table(pivot_acc)

plot_accuracy_bars(filtered_rew_df, save_prefix="rm_accuracy")

ppo_Walker2d-v5_912391_comparative_912391_noise_0.5_nfeedback_5000_rt1.0
Walker2d-v5 comparative 0.5 True
ppo_Walker2d-v5_912391_comparative_912391_noise_0.5_nfeedback_5000
Walker2d-v5 comparative 0.5 False
ppo_Swimmer-v5_912391_comparative_912391_noise_0.25_nfeedback_5000
Swimmer-v5 comparative 0.25 False
ppo_Swimmer-v5_912391_comparative_912391_noise_0.5_nfeedback_5000
Swimmer-v5 comparative 0.5 False
ppo_Swimmer-v5_912391_comparative_912391_noise_0.25_nfeedback_5000_rt1.0
Swimmer-v5 comparative 0.25 True
ppo_Swimmer-v5_912391_comparative_912391_noise_0.5_nfeedback_5000_rt1.0
Swimmer-v5 comparative 0.5 True
ppo_Walker2d-v5_1687123_comparative_1687123_noise_0.5_nfeedback_5000_rt1.0
Walker2d-v5 comparative 0.5 True
ppo_Walker2d-v5_1687123_comparative_1687123_noise_0.25_nfeedback_5000
Walker2d-v5 comparative 0.25 False
ppo_HalfCheetah-v5_912391_comparative_912391_noise_0.25_nfeedback_5000
HalfCheetah-v5 comparative 0.25 False
ppo_Walker2d-v5_1687123_comparative_1687123_noise_0.5_nfeedba